In [40]:
# Standard library imports
import sys
from pathlib import Path

# Third-party imports
import duckdb

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

## 1. Load & Clean with DuckDB

**Optimization:** Using **DuckDB** for out-of-core SQL processing. This allows us to query and clean the Parquet file directly from disk without loading it all into RAM, preventing kernel crashes.

In [41]:
# Define paths
input_path = Path('../data/processed/integrated_raw.parquet')
output_path = Path('../data/processed/integrated_clean.parquet')

if not input_path.exists():
    raise FileNotFoundError(
        "Integrated dataset not found! Please run notebook 04_integrate_datasets.ipynb first."
    )

print("Initializing DuckDB connection...")
con = duckdb.connect(database=':memory:')

# Inspect schema to build dynamic query
print("Inspecting schema...")
columns_info = con.execute(f"DESCRIBE SELECT * FROM '{input_path}'").fetchall()
columns = [col[0] for col in columns_info]
print(f"✓ Found {len(columns)} columns")

Initializing DuckDB connection...
Inspecting schema...
✓ Found 23 columns


## 2. Construct Cleaning Query

We dynamically build a SQL query to:
1. Select columns (handling renames and drops)
2. Deduplicate rows (`DISTINCT`)

In [42]:
print("Constructing SQL query...")

# Identify redundant columns
crash_suffix_cols = [c for c in columns if c.endswith('_crash')]

select_clauses = []
dropped_cols = []
renamed_cols = []

# Track processed columns to avoid duplicates in selection
processed_cols = set()

# Handle suffix columns first
for crash_col in crash_suffix_cols:
    base_name = crash_col.replace('_crash', '')
    person_col = base_name + '_person'
    
    if person_col in columns:
        # We have both. In SQL we'll select the crash one as the base name
        select_clauses.append(f'"{crash_col}" AS "{base_name}"')
        renamed_cols.append(base_name)
        dropped_cols.append(person_col)
        
        processed_cols.add(crash_col)
        processed_cols.add(person_col)

# Add remaining columns
for col in columns:
    if col not in processed_cols:
        select_clauses.append(f'"{col}"')

# Check if UNIQUE_ID exists for proper deduplication
has_unique_id = 'UNIQUE_ID' in columns

if has_unique_id:
    # Use proper deduplication based on UNIQUE_ID (preserves all legitimate records)
    # ROW_NUMBER() keeps first occurrence of each unique person record
    query = f"""
        SELECT {','.join(select_clauses)}
        FROM (
            SELECT 
                {','.join(select_clauses)},
                ROW_NUMBER() OVER (PARTITION BY "UNIQUE_ID" ORDER BY "CRASH_DATE") as rn
            FROM '{input_path}'
        ) sub
        WHERE rn = 1
    """
    dedup_method = "ROW_NUMBER() by UNIQUE_ID"
else:
    # Fallback to DISTINCT if no unique identifier exists
    query = f"""
        SELECT DISTINCT
            {','.join(select_clauses)}
        FROM '{input_path}'
    """
    dedup_method = "DISTINCT (all columns)"

print(f"✓ Plan created:")
print(f"  - Dropping {len(dropped_cols)} redundant columns")
print(f"  - Renaming {len(renamed_cols)} columns")
print(f"  - Deduplication method: {dedup_method}")

Constructing SQL query...
✓ Plan created:
  - Dropping 0 redundant columns
  - Renaming 0 columns
  - Deduplication method: ROW_NUMBER() by UNIQUE_ID


## 3. Execute and Save

Use DuckDB's `COPY` command to stream the result of the query directly to a Parquet file.

In [38]:
print("Executing pipeline and saving to disk...")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Execute COPY command
copy_query = f"COPY ({query}) TO '{output_path}' (FORMAT PARQUET, COMPRESSION 'SNAPPY')"
con.execute(copy_query)

print("✅ CLEANED INTEGRATED DATASET SAVED!")
print("="*60)
print(f"Path: {output_path}")
print(f"Size: {output_path.stat().st_size / (1024**2):.2f} MB")

# Verify
count = con.execute(f"SELECT COUNT(*) FROM '{output_path}'").fetchone()[0]
print(f"Records: {count:,}")

con.close()

Executing pipeline and saving to disk...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ CLEANED INTEGRATED DATASET SAVED!
Path: ..\data\processed\integrated_clean.parquet
Size: 380.63 MB
Records: 5,591,863
Records: 5,591,863


## 4. Data Validation Checks

Validate the cleaned data to ensure quality and catch any anomalies.

In [5]:
print("\n" + "="*60)
print("DATA VALIDATION CHECKS")
print("="*60)

# Reconnect to check the cleaned data
con = duckdb.connect(database=':memory:')

# Validation 1: Date Range (2012-2025)
print("\n1. Date Range Validation...")
date_col = None
for col_name in ['CRASH DATE', 'CRASH_DATE', 'crash_date']:
    if col_name in columns:
        date_col = col_name
        break

if date_col:
    invalid_dates_query = f"""
        SELECT COUNT(*) as invalid_count
        FROM '{output_path}'
        WHERE "{date_col}" < '2012-01-01' OR "{date_col}" > '2025-12-31'
    """
    invalid_date_count = con.execute(invalid_dates_query).fetchone()[0]
    print(f"   Records with invalid dates (outside 2012-2025): {invalid_date_count:,}")
    
    if invalid_date_count > 0:
        print(f"   ⚠️ Warning: {invalid_date_count} records have dates outside expected range")
    else:
        print("   ✅ All dates within valid range")
else:
    print("   ⚠️ Date column not found for validation")

# Validation 2: NYC Coordinate Boundaries
print("\n2. Geographic Coordinate Validation...")
# NYC bounds: 40.4774° to 40.9176° N, -74.2591° to -73.7004° W
lat_col = None
lon_col = None
for lat_name in ['LATITUDE', 'latitude']:
    if lat_name in columns:
        lat_col = lat_name
        break
for lon_name in ['LONGITUDE', 'longitude']:
    if lon_name in columns:
        lon_col = lon_name
        break

if lat_col and lon_col:
    invalid_coords_query = f"""
        SELECT COUNT(*) as invalid_count
        FROM '{output_path}'
        WHERE (
            "{lat_col}" IS NOT NULL AND "{lon_col}" IS NOT NULL
            AND (
                "{lat_col}" < 40.4774 OR "{lat_col}" > 40.9176 OR
                "{lon_col}" < -74.2591 OR "{lon_col}" > -73.7004
            )
        )
    """
    invalid_coord_count = con.execute(invalid_coords_query).fetchone()[0]
    print(f"   Records with coordinates outside NYC bounds: {invalid_coord_count:,}")
    
    if invalid_coord_count > 0:
        print(f"   ⚠️ Warning: {invalid_coord_count} records have invalid coordinates")
    else:
        print("   ✅ All coordinates within NYC boundaries")
else:
    print("   ⚠️ Coordinate columns not found for validation")

# Validation 3: Casualty Sum Check
print("\n3. Casualty Sum Validation...")
injured_col = None
killed_col = None
total_col = None

for inj_name in ['NUMBER OF PERSONS INJURED', 'NUMBER_OF_PERSONS_INJURED', 'total_injured']:
    if inj_name in columns:
        injured_col = inj_name
        break
for kill_name in ['NUMBER OF PERSONS KILLED', 'NUMBER_OF_PERSONS_KILLED', 'total_killed']:
    if kill_name in columns:
        killed_col = kill_name
        break
for tot_name in ['total_casualties', 'TOTAL_CASUALTIES']:
    if tot_name in columns:
        total_col = tot_name
        break

if injured_col and killed_col and total_col:
    casualty_check_query = f"""
        SELECT COUNT(*) as mismatch_count
        FROM '{output_path}'
        WHERE COALESCE("{injured_col}", 0) + COALESCE("{killed_col}", 0) != COALESCE("{total_col}", 0)
    """
    mismatch_count = con.execute(casualty_check_query).fetchone()[0]
    print(f"   Records with casualty sum mismatches: {mismatch_count:,}")
    
    if mismatch_count > 0:
        print(f"   ⚠️ Warning: {mismatch_count} records have incorrect casualty totals")
    else:
        print("   ✅ All casualty sums are correct")
else:
    print("   ⚠️ Casualty columns not found for validation")

# Summary
print("\n" + "="*60)
print("✅ Validation checks complete!")
print("="*60)

con.close()


DATA VALIDATION CHECKS

1. Date Range Validation...
   Records with invalid dates (outside 2012-2025): 0
   ✅ All dates within valid range

2. Geographic Coordinate Validation...
   Records with coordinates outside NYC bounds: 21,273
   ⚠️ Warning: 21273 records have invalid coordinates

3. Casualty Sum Validation...
   ⚠️ Casualty columns not found for validation

✅ Validation checks complete!
   Records with coordinates outside NYC bounds: 21,273
   ⚠️ Warning: 21273 records have invalid coordinates

3. Casualty Sum Validation...
   ⚠️ Casualty columns not found for validation

✅ Validation checks complete!


In [39]:
print("\n" + "="*60)
print("DEATH COUNT INVESTIGATION")
print("="*60)

# Reconnect for death count investigation
con = duckdb.connect(database=':memory:')

# Check deaths at each stage of the pipeline
print("\n🔍 Tracing death counts through pipeline stages...\n")

# 1. Check original crashes dataset
crashes_path = Path('../data/processed/01_cleaned_crashes.parquet')
if crashes_path.exists():
    crashes_deaths = con.execute(f"""
        SELECT SUM("NUMBER OF PERSONS KILLED") as total_deaths
        FROM '{crashes_path}'
    """).fetchone()[0]
    print(f"1. Original Crashes Dataset: {crashes_deaths:,} deaths")
else:
    print("1. Original Crashes Dataset: NOT FOUND")

# 2. Check original persons dataset
persons_path = Path('../data/processed/02_cleaned_persons.parquet')
if persons_path.exists():
    persons_deaths = con.execute(f"""
        SELECT COUNT(*) as total_deaths
        FROM '{persons_path}'
        WHERE "PERSON_INJURY" = 'Killed'
    """).fetchone()[0]
    print(f"2. Cleaned Persons Dataset: {persons_deaths:,} deaths")
else:
    print("2. Cleaned Persons Dataset: NOT FOUND")

# 3. Check integrated_raw (before DISTINCT)
integrated_raw_path = Path('../data/processed/integrated_raw.parquet')
if integrated_raw_path.exists():
    raw_deaths = con.execute(f"""
        SELECT COUNT(*) as total_deaths
        FROM '{integrated_raw_path}'
        WHERE "PERSON_INJURY" = 'Killed'
    """).fetchone()[0]
    print(f"3. After Integration (before DISTINCT): {raw_deaths:,} deaths")
else:
    print("3. After Integration: NOT FOUND")
    raw_deaths = None

# 4. Check integrated_clean (after DISTINCT)
clean_deaths = con.execute(f"""
    SELECT COUNT(*) as total_deaths
    FROM '{output_path}'
    WHERE "PERSON_INJURY" = 'Killed'
""").fetchone()[0]
print(f"4. After DISTINCT Deduplication: {clean_deaths:,} deaths")

# 5. Calculate losses at each stage
print("\n" + "="*60)
print("DEATH COUNT LOSSES BY STAGE:")
print("="*60)

if crashes_path.exists():
    crash_total = crashes_deaths if 'crashes_deaths' in locals() else 0
    print(f"\n📊 Starting Point (Crashes Dataset): {crash_total:,} deaths")

if persons_path.exists():
    if 'persons_deaths' in locals():
        person_total = persons_deaths
        if 'crashes_deaths' in locals():
            loss_in_cleaning = crashes_deaths - persons_deaths
            pct_loss_cleaning = (loss_in_cleaning / crashes_deaths * 100) if crashes_deaths > 0 else 0
            print(f"   Loss in Notebook 03 (age outlier removal): -{loss_in_cleaning:,} ({pct_loss_cleaning:.1f}%)")
            print(f"   → Remaining after person cleaning: {person_total:,} deaths")

if raw_deaths is not None:
    if 'persons_deaths' in locals():
        loss_in_integration = persons_deaths - raw_deaths
        pct_loss_integration = (loss_in_integration / persons_deaths * 100) if persons_deaths > 0 else 0
        if loss_in_integration != 0:
            print(f"   Loss in Notebook 04 (integration): -{loss_in_integration:,} ({pct_loss_integration:.1f}%)")
        print(f"   → Remaining after integration: {raw_deaths:,} deaths")
    
    loss_in_distinct = raw_deaths - clean_deaths
    pct_loss_distinct = (loss_in_distinct / raw_deaths * 100) if raw_deaths > 0 else 0
    print(f"   Loss in Notebook 05 (DISTINCT): -{loss_in_distinct:,} ({pct_loss_distinct:.1f}%)")

print(f"\n🎯 Final Death Count: {clean_deaths:,}")
print(f"   Expected (NYC Official): 3,000-3,500")

if 'crashes_deaths' in locals():
    total_loss = crashes_deaths - clean_deaths
    total_pct = (total_loss / crashes_deaths * 100) if crashes_deaths > 0 else 0
    print(f"   Total Loss from Original: -{total_loss:,} ({total_pct:.1f}%)")

print("\n" + "="*60)


DEATH COUNT INVESTIGATION

🔍 Tracing death counts through pipeline stages...

1. Original Crashes Dataset: 3,517.0 deaths
2. Cleaned Persons Dataset: 3,516 deaths
3. After Integration (before DISTINCT): 3,516 deaths
4. After DISTINCT Deduplication: 3,516 deaths

DEATH COUNT LOSSES BY STAGE:

📊 Starting Point (Crashes Dataset): 3,517.0 deaths
   Loss in Notebook 03 (age outlier removal): -1.0 (0.0%)
   → Remaining after person cleaning: 3,516 deaths
   → Remaining after integration: 3,516 deaths
   Loss in Notebook 05 (DISTINCT): -0 (0.0%)

🎯 Final Death Count: 3,516
   Expected (NYC Official): 3,000-3,500
   Total Loss from Original: -1.0 (0.0%)

